# 01 — Data Acquisition (MIMIC-IV Demo)

This notebook downloads the open-access **MIMIC-IV Clinical Database Demo (v2.2)**, loads its relational tables, and reshapes them into the **canonical hourly time-series** used by the rest of the pipeline.

> Replaces the former PhysioNet-2019 flat-file loader. The 2019 Challenge set has no urine output, which makes full KDIGO AKI staging impossible. The MIMIC-IV demo has `icu/outputevents`, so we can build the complete creatinine **and** urine-output criteria downstream.

The demo is **de-identified and open-access** — no PhysioNet credentialing is required (that gate applies only to the full MIMIC-IV).

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
!pip install -q pandas numpy pyarrow tqdm scipy scikit-learn xgboost lightgbm shap matplotlib seaborn imbalanced-learn optuna boto3

## Download the MIMIC-IV Demo

The demo is mirrored on the **PhysioNet open-data S3 bucket** (`physionet-open/mimic-iv-demo/`) and is downloaded anonymously — no AWS account or PhysioNet login needed. The cell below is idempotent: it skips any file already present, so it is safe to re-run.

The data lands in `../data/raw/mimic_iv_demo/`, preserving the `hosp/` and `icu/` folder structure that `data_loader` expects.

In [ ]:
# import os, sys
# # point this at YOUR clone location if different
# PROJ = '/content/drive/MyDrive/sentinel-poc/project_sentinel'
# os.chdir(os.path.join(PROJ, 'notebooks'))   # so ../data/... paths resolve
# sys.path.insert(0, PROJ)                      # so `from src...` works
# print('cwd =', os.getcwd())

cwd = /content/drive/MyDrive/sentinel-poc/project_sentinel/notebooks


In [10]:
import os

try:
    import boto3
except ImportError:
    !pip install -q boto3
    import boto3
from botocore import UNSIGNED
from botocore.config import Config

dest = "../data/raw/mimic_iv_demo"
os.makedirs(dest, exist_ok=True)

s3 = boto3.client("s3", config=Config(signature_version=UNSIGNED))
bucket, prefix = "physionet-open", "mimic-iv-demo/"

paginator = s3.get_paginator("list_objects_v2")
n_new = 0
for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
    for obj in page.get("Contents", []):
        key = obj["Key"]
        if key.endswith("/"):
            continue
        local = os.path.join(dest, os.path.relpath(key, prefix))
        os.makedirs(os.path.dirname(local), exist_ok=True)
        if not os.path.exists(local):
            s3.download_file(bucket, key, local)
            n_new += 1

print(f"✓ MIMIC-IV demo ready in {dest} ({n_new} new files downloaded).")

✓ MIMIC-IV demo ready in ../data/raw/mimic_iv_demo (0 new files downloaded).


In [11]:
import sys
from pathlib import Path

# Add the project root to sys.path so we can import from src
sys.path.append(str(Path.cwd().parent))

from src.data_loader import (
    load_mimic_demo,
    build_hourly_cohort,
    identify_sepsis_onset,
    print_data_summary,
)

data_dir = "../data/raw/mimic_iv_demo"
print(f"Data directory set to: {data_dir}")

print("Loading MIMIC-IV demo tables...")
tables = load_mimic_demo(data_dir)
for name, t in tables.items():
    print(f"  {name:20s} {len(t):>9,} rows")

Data directory set to: ../data/raw/mimic_iv_demo
Loading MIMIC-IV demo tables...
  patients                   100 rows
  icustays                   140 rows
  chartevents            668,862 rows
  labevents              107,727 rows
  outputevents             9,362 rows
  prescriptions           18,087 rows
  microbiologyevents       2,899 rows


In [ ]:
import pandas as pd

print("Building hourly time-series cohort (one row per ICU stay-hour)...")
hourly = build_hourly_cohort(tables, data_dir)

print("\nIdentifying sepsis onset (Sepsis-3 suspicion-of-infection proxy)...")
hourly = identify_sepsis_onset(hourly, tables)

print()
print_data_summary(hourly)
display(hourly.head())

# Persist the raw hourly frame for the cohort-definition notebook (02)
interim = Path("../data/interim")
interim.mkdir(parents=True, exist_ok=True)
hourly.to_parquet(interim / "hourly_cohort.parquet", index=False)
print(f"\nSaved → {interim / 'hourly_cohort.parquet'}  (shape={hourly.shape})")

## Conclusion

The MIMIC-IV demo has been downloaded, its relational tables loaded, and reshaped into the canonical hourly time-series (`hourly_cohort.parquet`) — including the **urine-output rate and its observation mask**, which the downstream KDIGO labeling relies on.

**Next:** `02_cohort_definition.ipynb` applies inclusion criteria, computes baseline creatinine, and excludes prevalent AKI to produce the final modeling cohort.

> ⚠️ Sepsis onset currently uses the *suspicion-of-infection* half of Sepsis-3 (antibiotic + culture pairing). The SOFA ≥ 2 component is a documented simplification for the demo — see `identify_sepsis_onset` in `src/data_loader.py`.

In [16]:
!git config --global user.email "dcsenga442@gmail.com"
!git config --global user.name  "Sng43"

from getpass import getpass
tok = getpass("GitHub PAT: ")
# We use the token to set the remote, then immediately clear it from memory
!cd /content/drive/MyDrive/sentinel-poc && git remote set-url origin https://{tok}@github.com/Sng43/sentinel-poc.git

# Clear the variable and the cell output manually to ensure it isn't saved in the .ipynb
tok = None
print("Remote updated. Token cleared from memory.")

GitHub PAT: ··········
Remote updated. Token cleared from memory.


In [24]:
!git pull

hint: You have divergent branches and need to specify how to reconcile them.
hint: You can do so by running one of the following commands sometime before
hint: your next pull:
hint: 
hint:   git config pull.rebase false  # merge (the default strategy)
hint:   git config pull.rebase true   # rebase
hint:   git config pull.ff only       # fast-forward only
hint: 
hint: You can replace "git config" with "git config --global" to set a default
hint: preference for all repositories. You can also pass --rebase, --no-rebase,
hint: or --ff-only on the command line to override the configured default per
hint: invocation.
fatal: Need to specify how to reconcile divergent branches.


### Verify Git Configuration
Run the cell below to check if your remote URL now correctly includes your token (masked) and that the project path is valid.

In [25]:
%cd /content/drive/MyDrive/sentinel-poc

# 1. Reconcile with the remote using rebase
# This brings in any remote changes and puts your clean work on top
print("Reconciling history...")
!git pull --rebase origin main

# 2. Add and commit your current clean state
!git add -A
!git commit -m "run: 01_data_acquisition (clean)"

# 3. Force push to overwrite the 'bad' history on GitHub with the 'clean' one
print("Pushing clean history to GitHub...")
!git push origin main --force

/content/drive/MyDrive/sentinel-poc
Reconciling history...
error: cannot pull with rebase: You have unstaged changes.
error: please commit or stash them.
[main bf1a604] run: 01_data_acquisition (clean)
 1 file changed, 1 insertion(+), 1 deletion(-)
Pushing clean history to GitHub...
Enumerating objects: 27, done.
Counting objects: 100% (27/27), done.
Delta compression using up to 2 threads
Compressing objects: 100% (16/16), done.
Writing objects: 100% (19/19), 11.58 KiB | 515.00 KiB/s, done.
Total 19 (delta 14), reused 3 (delta 3), pack-reused 0
remote: Resolving deltas: 100% (14/14), completed with 6 local objects.
To https://github.com/Sng43/sentinel-poc.git
 + c553e54...bf1a604 main -> main (forced update)
